# Sortformer versus DiaPer overlap test

Attach the completed `diarization-results(15).zip`. This notebook reuses its preserved baseline and full-video audio, runs NVIDIA's offline four-speaker Sortformer in bounded contextual windows, and applies the same overlap/control comparison used for DiaPer. It does not rerun or modify the transcription pipeline. The public model is licensed CC-BY-NC-4.0.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

REVISION = "sortformer-extracted-dataset-v3"
VIDEO_ID = "lVfKfbFd0SM"
NEMO_COMMIT = "429e2ac4b69398d3cefb0ed15dbf9c043f290367"
BASE = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path.cwd()/"sortformer-run"
WORK = BASE/"sortformer-comparison"
PRIOR = BASE/"prior-diarization-results"
OUTPUT = BASE/"sortformer-results"
VENV = BASE/"sortformer-venv"
PYTHON = str(VENV/"bin"/"python")
for directory in (WORK, PRIOR, OUTPUT): directory.mkdir(parents=True, exist_ok=True)
print("Revision:", REVISION)
print("Output:", OUTPUT)


## Install the isolated Sortformer runtime

Enable Internet and a GPU. NeMo is pinned to the recorded commit for reproducibility.


In [ ]:
EMBEDDED_FILES = {'evaluate_diaper_overlap.py': '"""Compare a supplemental DiaPer RTTM with the preserved baseline '
                               'evidence.\n'
                               '\n'
                               "This evaluator never changes the baseline.  It maps DiaPer's "
                               'anonymous speaker\n'
                               'labels to baseline tracks using clean, single-speaker time and '
                               'then measures\n'
                               'whether DiaPer independently corroborates the baseline overlap '
                               'intervals.\n'
                               '"""\n'
                               '\n'
                               'import argparse\n'
                               'from collections import defaultdict\n'
                               'import json\n'
                               'from pathlib import Path\n'
                               '\n'
                               '\n'
                               'def parse_rttm(path):\n'
                               '    rows = []\n'
                               '    for line in Path(path).read_text().splitlines():\n'
                               '        fields = line.split()\n'
                               '        if not fields or fields[0] != "SPEAKER" or len(fields) < '
                               '8:\n'
                               '            continue\n'
                               '        start, duration = float(fields[3]), float(fields[4])\n'
                               '        rows.append({"start": start, "end": start + duration, '
                               '"speaker": fields[7]})\n'
                               '    return rows\n'
                               '\n'
                               '\n'
                               'def overlap_duration(left_start, left_end, right_start, '
                               'right_end):\n'
                               '    return max(0.0, min(left_end, right_end) - max(left_start, '
                               'right_start))\n'
                               '\n'
                               '\n'
                               'def overlap_evidence(segment):\n'
                               '    return next((\n'
                               '        item for item in segment.get("evidence", [])\n'
                               '        if item.get("source") == "overlapping_speakers"\n'
                               '        and item.get("details", {}).get("target_and_non_target", '
                               'False)\n'
                               '    ), None)\n'
                               '\n'
                               '\n'
                               'def map_speakers(segments, rttm):\n'
                               '    """Greedily map anonymous DiaPer speakers from clean baseline '
                               'intersections."""\n'
                               '    scores = defaultdict(float)\n'
                               '    for segment in segments:\n'
                               '        if overlap_evidence(segment):\n'
                               '            continue\n'
                               '        baseline = segment.get("baseline", {})\n'
                               '        track = baseline.get("raw_speaker_track")\n'
                               '        if not track:\n'
                               '            continue\n'
                               '        for row in rttm:\n'
                               '            scores[(row["speaker"], track)] += overlap_duration(\n'
                               '                segment["start"], segment["end"], row["start"], '
                               'row["end"]\n'
                               '            )\n'
                               '    candidates = sorted(\n'
                               '        ((seconds, diaper, track) for (diaper, track), seconds in '
                               'scores.items()),\n'
                               '        reverse=True,\n'
                               '    )\n'
                               '    mapping, used_tracks = {}, set()\n'
                               '    for seconds, diaper, track in candidates:\n'
                               '        if diaper not in mapping and track not in used_tracks and '
                               'seconds > 0:\n'
                               '            mapping[diaper] = {"baseline_track": track, '
                               '"intersection_seconds": seconds}\n'
                               '            used_tracks.add(track)\n'
                               '    return mapping\n'
                               '\n'
                               '\n'
                               'def activity_summary(start, end, rttm, step=0.02):\n'
                               '    duration = max(end - start, 1e-9)\n'
                               '    frame_count = max(1, int(duration / step + 0.999))\n'
                               '    active_sets = []\n'
                               '    for index in range(frame_count):\n'
                               '        time = min(end, start + (index + 0.5) * duration / '
                               'frame_count)\n'
                               '        active_sets.append({\n'
                               '            row["speaker"] for row in rttm if row["start"] <= time '
                               '< row["end"]\n'
                               '        })\n'
                               '    overlap_frames = sum(len(active) >= 2 for active in '
                               'active_sets)\n'
                               '    return {\n'
                               '        "overlap_fraction": overlap_frames / frame_count,\n'
                               '        "maximum_simultaneous_speakers": max(map(len, '
                               'active_sets), default=0),\n'
                               '        "active_speakers": sorted(set().union(*active_sets) if '
                               'active_sets else set()),\n'
                               '    }\n'
                               '\n'
                               '\n'
                               'def evaluate(baseline, rttm, minimum_overlap_fraction=0.10):\n'
                               '    segments = baseline.get("segments", [])\n'
                               '    mapping = map_speakers(segments, rttm)\n'
                               '    target_track = baseline.get("target_candidate")\n'
                               '    target_diaper = next((\n'
                               '        speaker for speaker, item in mapping.items()\n'
                               '        if item["baseline_track"] == target_track\n'
                               '    ), None)\n'
                               '    rows = []\n'
                               '    for index, segment in enumerate(segments):\n'
                               '        summary = activity_summary(float(segment["start"]), '
                               'float(segment["end"]), rttm)\n'
                               '        expected = overlap_evidence(segment) is not None\n'
                               '        detected = summary["overlap_fraction"] >= '
                               'minimum_overlap_fraction\n'
                               '        active_mapped = [mapping.get(speaker, '
                               '{}).get("baseline_track")\n'
                               '                         for speaker in '
                               'summary["active_speakers"]]\n'
                               '        rows.append({\n'
                               '            "baseline_index": index,\n'
                               '            "start": segment["start"],\n'
                               '            "end": segment["end"],\n'
                               '            "text": segment.get("text", ""),\n'
                               '            "baseline_speaker": segment.get("final_speaker"),\n'
                               '            "baseline_target_non_target_overlap": expected,\n'
                               '            "diaper_overlap_detected": detected,\n'
                               '            "diaper_target_and_other_active": bool(\n'
                               '                target_diaper in summary["active_speakers"]\n'
                               '                and len(summary["active_speakers"]) >= 2\n'
                               '            ),\n'
                               '            "mapped_active_tracks": sorted(x for x in '
                               'active_mapped if x),\n'
                               '            **summary,\n'
                               '        })\n'
                               '    expected_rows = [row for row in rows if '
                               'row["baseline_target_non_target_overlap"]]\n'
                               '    controls = [row for row in rows if not '
                               'row["baseline_target_non_target_overlap"]]\n'
                               '    summary = {\n'
                               '        "baseline_segments": len(rows),\n'
                               '        "baseline_overlap_segments": len(expected_rows),\n'
                               '        "diaper_corroborated_overlap_segments": sum(\n'
                               '            row["diaper_overlap_detected"] for row in '
                               'expected_rows\n'
                               '        ),\n'
                               '        "diaper_target_plus_other_segments": sum(\n'
                               '            row["diaper_target_and_other_active"] for row in '
                               'expected_rows\n'
                               '        ),\n'
                               '        "non_overlap_control_segments": len(controls),\n'
                               '        "diaper_overlap_on_control_segments": sum(\n'
                               '            row["diaper_overlap_detected"] for row in controls\n'
                               '        ),\n'
                               '        "minimum_overlap_fraction": minimum_overlap_fraction,\n'
                               '        "target_baseline_track": target_track,\n'
                               '        "target_diaper_speaker": target_diaper,\n'
                               '        "speaker_mapping": mapping,\n'
                               '        "interpretation": (\n'
                               '            "Agreement is corroborating evidence only; without '
                               'hand labels it is not accuracy."\n'
                               '        ),\n'
                               '    }\n'
                               '    return {"summary": summary, "segments": rows}\n'
                               '\n'
                               '\n'
                               'def main():\n'
                               '    parser = argparse.ArgumentParser()\n'
                               '    parser.add_argument("--baseline", type=Path, required=True)\n'
                               '    parser.add_argument("--rttm", type=Path, required=True)\n'
                               '    parser.add_argument("--output", type=Path, required=True)\n'
                               '    parser.add_argument("--minimum-overlap-fraction", type=float, '
                               'default=0.10)\n'
                               '    args = parser.parse_args()\n'
                               '    result = evaluate(\n'
                               '        json.loads(args.baseline.read_text()), '
                               'parse_rttm(args.rttm),\n'
                               '        args.minimum_overlap_fraction,\n'
                               '    )\n'
                               '    args.output.parent.mkdir(parents=True, exist_ok=True)\n'
                               '    args.output.write_text(json.dumps(result, indent=2) + "\\n")\n'
                               '    print(json.dumps(result["summary"], indent=2))\n'
                               '\n'
                               '\n'
                               'if __name__ == "__main__":\n'
                               '    main()\n',
 'evaluate_overlap_activity.py': '"""Compare any overlap-aware RTTM with preserved baseline '
                                 'overlap evidence."""\n'
                                 '\n'
                                 'import argparse\n'
                                 'import json\n'
                                 'from pathlib import Path\n'
                                 '\n'
                                 'from evaluate_diaper_overlap import activity_summary, '
                                 'overlap_evidence, parse_rttm\n'
                                 '\n'
                                 '\n'
                                 'def evaluate(baseline, rttm, threshold=0.10):\n'
                                 '    rows = []\n'
                                 '    for index, segment in enumerate(baseline.get("segments", '
                                 '[])):\n'
                                 '        activity = activity_summary(float(segment["start"]), '
                                 'float(segment["end"]), rttm)\n'
                                 '        expected = overlap_evidence(segment) is not None\n'
                                 '        rows.append({\n'
                                 '            "baseline_index": index,\n'
                                 '            "start": segment["start"], "end": segment["end"],\n'
                                 '            "text": segment.get("text", ""),\n'
                                 '            "baseline_speaker": segment.get("final_speaker"),\n'
                                 '            "baseline_target_non_target_overlap": expected,\n'
                                 '            "system_overlap_detected": '
                                 'activity["overlap_fraction"] >= threshold,\n'
                                 '            **activity,\n'
                                 '        })\n'
                                 '    positives = [row for row in rows if '
                                 'row["baseline_target_non_target_overlap"]]\n'
                                 '    controls = [row for row in rows if not '
                                 'row["baseline_target_non_target_overlap"]]\n'
                                 '    sweep = []\n'
                                 '    for value in (0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, '
                                 '0.50):\n'
                                 '        tp = sum(row["overlap_fraction"] >= value for row in '
                                 'positives)\n'
                                 '        fp = sum(row["overlap_fraction"] >= value for row in '
                                 'controls)\n'
                                 '        precision = tp / max(1, tp + fp)\n'
                                 '        recall = tp / max(1, len(positives))\n'
                                 '        sweep.append({\n'
                                 '            "threshold": value, "corroborated": tp, '
                                 '"control_detections": fp,\n'
                                 '            "agreement_precision": precision, '
                                 '"agreement_recall": recall,\n'
                                 '            "agreement_f1": 2 * precision * recall / max(1e-9, '
                                 'precision + recall),\n'
                                 '        })\n'
                                 '    chosen = next(item for item in sweep if item["threshold"] == '
                                 'threshold)\n'
                                 '    return {\n'
                                 '        "summary": {\n'
                                 '            "baseline_segments": len(rows),\n'
                                 '            "baseline_overlap_segments": len(positives),\n'
                                 '            "corroborated_overlap_segments": '
                                 'chosen["corroborated"],\n'
                                 '            "non_overlap_control_segments": len(controls),\n'
                                 '            "overlap_on_control_segments": '
                                 'chosen["control_detections"],\n'
                                 '            "minimum_overlap_fraction": threshold,\n'
                                 '            "agreement_precision": '
                                 'chosen["agreement_precision"],\n'
                                 '            "agreement_recall": chosen["agreement_recall"],\n'
                                 '            "agreement_f1": chosen["agreement_f1"],\n'
                                 '            "interpretation": "Agreement between imperfect '
                                 'systems is not measured accuracy.",\n'
                                 '        },\n'
                                 '        "threshold_sweep": sweep,\n'
                                 '        "segments": rows,\n'
                                 '    }\n'
                                 '\n'
                                 '\n'
                                 'def main():\n'
                                 '    parser = argparse.ArgumentParser()\n'
                                 '    parser.add_argument("--baseline", type=Path, required=True)\n'
                                 '    parser.add_argument("--rttm", type=Path, required=True)\n'
                                 '    parser.add_argument("--output", type=Path, required=True)\n'
                                 '    args = parser.parse_args()\n'
                                 '    result = evaluate(json.loads(args.baseline.read_text()), '
                                 'parse_rttm(args.rttm))\n'
                                 '    args.output.parent.mkdir(parents=True, exist_ok=True)\n'
                                 '    args.output.write_text(json.dumps(result, indent=2) + '
                                 '"\\n")\n'
                                 '    print(json.dumps(result["summary"], indent=2))\n'
                                 '\n'
                                 '\n'
                                 'if __name__ == "__main__":\n'
                                 '    main()\n',
 'run_sortformer_overlap.py': '"""Run offline Sortformer in bounded, contextual windows and emit '
                              'RTTM.\n'
                              '\n'
                              'Speaker labels are intentionally scoped to each window.  This '
                              'experiment uses\n'
                              'Sortformer as an independent overlap detector, so cross-window '
                              'identity is not\n'
                              'assumed or needed.\n'
                              '"""\n'
                              '\n'
                              'import argparse\n'
                              'import json\n'
                              'from pathlib import Path\n'
                              'import subprocess\n'
                              '\n'
                              '\n'
                              'def parse_segment(row):\n'
                              '    if isinstance(row, str):\n'
                              '        fields = row.replace(",", " ").split()\n'
                              '    elif isinstance(row, (list, tuple)):\n'
                              '        fields = list(row)\n'
                              '    else:\n'
                              '        raise TypeError(f"Unsupported Sortformer segment: '
                              '{type(row).__name__}: {row!r}")\n'
                              '    if len(fields) < 3:\n'
                              '        raise ValueError(f"Incomplete Sortformer segment: '
                              '{row!r}")\n'
                              '    return float(fields[0]), float(fields[1]), str(fields[2])\n'
                              '\n'
                              '\n'
                              'def unwrap(result):\n'
                              '    while isinstance(result, list) and len(result) == 1 and '
                              'isinstance(result[0], list):\n'
                              '        result = result[0]\n'
                              '    return result\n'
                              '\n'
                              '\n'
                              'def main():\n'
                              '    parser = argparse.ArgumentParser()\n'
                              '    parser.add_argument("--audio", type=Path, required=True)\n'
                              '    parser.add_argument("--output-dir", type=Path, required=True)\n'
                              '    parser.add_argument("--model", '
                              'default="nvidia/diar_sortformer_4spk-v1")\n'
                              '    parser.add_argument("--core-seconds", type=float, '
                              'default=80.0)\n'
                              '    parser.add_argument("--context-seconds", type=float, '
                              'default=5.0)\n'
                              '    args = parser.parse_args()\n'
                              '\n'
                              '    import soundfile as sf\n'
                              '    import torch\n'
                              '    from nemo.collections.asr.models import '
                              'SortformerEncLabelModel\n'
                              '\n'
                              '    if not torch.cuda.is_available():\n'
                              '        raise RuntimeError("Sortformer comparison requires a CUDA '
                              'GPU")\n'
                              '    args.output_dir.mkdir(parents=True, exist_ok=True)\n'
                              '    crop_dir = args.output_dir / "windows"\n'
                              '    crop_dir.mkdir(exist_ok=True)\n'
                              '    duration = float(sf.info(args.audio).duration)\n'
                              '    model = SortformerEncLabelModel.from_pretrained(args.model)\n'
                              '    model = model.to("cuda").eval()\n'
                              '\n'
                              '    rttm_rows, manifest = [], []\n'
                              '    core_start, window_index = 0.0, 0\n'
                              '    while core_start < duration:\n'
                              '        core_end = min(duration, core_start + args.core_seconds)\n'
                              '        crop_start = max(0.0, core_start - args.context_seconds)\n'
                              '        crop_end = min(duration, core_end + args.context_seconds)\n'
                              '        crop = crop_dir / f"window-{window_index:03d}.wav"\n'
                              '        subprocess.run([\n'
                              '            "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", '
                              '"error", "-y",\n'
                              '            "-ss", str(crop_start), "-i", str(args.audio),\n'
                              '            "-t", str(crop_end - crop_start), "-ac", "1", "-ar", '
                              '"16000", str(crop),\n'
                              '        ], check=True)\n'
                              '        predicted = unwrap(model.diarize(audio=str(crop), '
                              'batch_size=1))\n'
                              '        kept = 0\n'
                              '        for row in predicted:\n'
                              '            local_start, local_end, speaker = parse_segment(row)\n'
                              '            absolute_start, absolute_end = local_start + '
                              'crop_start, local_end + crop_start\n'
                              '            start, end = max(core_start, absolute_start), '
                              'min(core_end, absolute_end)\n'
                              '            if end <= start:\n'
                              '                continue\n'
                              '            scoped_speaker = f"window{window_index:03d}_{speaker}"\n'
                              '            rttm_rows.append(\n'
                              '                f"SPEAKER full-video 1 {start:.3f} {end-start:.3f} '
                              '<NA> <NA> {scoped_speaker} <NA> <NA>"\n'
                              '            )\n'
                              '            kept += 1\n'
                              '        manifest.append({\n'
                              '            "window": window_index, "core_start": core_start, '
                              '"core_end": core_end,\n'
                              '            "crop_start": crop_start, "crop_end": crop_end, '
                              '"segments_kept": kept,\n'
                              '        })\n'
                              '        print(f"Sortformer window {window_index + 1}: '
                              '{core_start:.1f}-{core_end:.1f}s, {kept} segments")\n'
                              '        core_start, window_index = core_end, window_index + 1\n'
                              '\n'
                              '    (args.output_dir / '
                              '"sortformer.rttm").write_text("\\n".join(rttm_rows) + "\\n")\n'
                              '    (args.output_dir / '
                              '"windows.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n'
                              '    print(f"Saved {len(rttm_rows)} RTTM rows across {len(manifest)} '
                              'windows")\n'
                              '\n'
                              '\n'
                              'if __name__ == "__main__":\n'
                              '    main()\n',
 'test_overlap_activity.py': 'import unittest\n'
                             '\n'
                             'from evaluate_overlap_activity import evaluate\n'
                             '\n'
                             '\n'
                             'class OverlapActivityTest(unittest.TestCase):\n'
                             '    def test_reports_overlap_and_control_agreement(self):\n'
                             '        overlap = [{"source": "overlapping_speakers", "details": '
                             '{"target_and_non_target": True}}]\n'
                             '        baseline = {"segments": [\n'
                             '            {"start": 0.0, "end": 1.0, "text": "both", "evidence": '
                             'overlap},\n'
                             '            {"start": 1.0, "end": 2.0, "text": "one", "evidence": '
                             '[]},\n'
                             '        ]}\n'
                             '        rttm = [\n'
                             '            {"start": 0.0, "end": 2.0, "speaker": "a"},\n'
                             '            {"start": 0.2, "end": 0.8, "speaker": "b"},\n'
                             '        ]\n'
                             '        result = evaluate(baseline, rttm)\n'
                             '        '
                             'self.assertEqual(result["summary"]["corroborated_overlap_segments"], '
                             '1)\n'
                             '        '
                             'self.assertEqual(result["summary"]["overlap_on_control_segments"], '
                             '0)\n'
                             '\n'
                             '\n'
                             'if __name__ == "__main__":\n'
                             '    unittest.main()\n',
 'test_sortformer_overlap.py': 'import unittest\n'
                               '\n'
                               'from run_sortformer_overlap import parse_segment, unwrap\n'
                               '\n'
                               '\n'
                               'class SortformerOutputTest(unittest.TestCase):\n'
                               '    def test_parses_documented_string_output(self):\n'
                               '        self.assertEqual(parse_segment("1.25 2.50 speaker_0"), '
                               '(1.25, 2.5, "speaker_0"))\n'
                               '\n'
                               '    def test_parses_sequence_and_unwraps_single_file_batch(self):\n'
                               '        self.assertEqual(parse_segment((1, 2, "speaker_1")), (1.0, '
                               '2.0, "speaker_1"))\n'
                               '        self.assertEqual(unwrap([["0 1 speaker_0"]]), ["0 1 '
                               'speaker_0"])\n'
                               '\n'
                               '\n'
                               'if __name__ == "__main__":\n'
                               '    unittest.main()\n'}
for name, source in EMBEDDED_FILES.items():
    (WORK/name).write_text(source)

ENV = os.environ.copy()
ENV["PYTHONUNBUFFERED"] = "1"
ENV["MPLBACKEND"] = "Agg"
ENV["HF_HOME"] = str(BASE/"huggingface-cache")
def checked(command, **kwargs):
    return subprocess.run(command, env=ENV, check=True, **kwargs)

if not Path(PYTHON).is_file():
    bootstrap = BASE/"sortformer-bootstrap"
    checked([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
             "virtualenv>=20.26,<21"])
    bootstrap_env = ENV.copy(); bootstrap_env["PYTHONPATH"] = str(bootstrap)
    subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages",
                    "--no-download", str(VENV)], env=bootstrap_env, check=True)

checked([PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
checked([PYTHON, "-m", "pip", "install", "Cython", "packaging", "soundfile"])
checked([PYTHON, "-m", "pip", "install",
         "nemo_toolkit[asr] @ git+https://github.com/NVIDIA-NeMo/NeMo.git@"+NEMO_COMMIT])
checked([PYTHON, "-c", "import torch; from nemo.collections.asr.models import SortformerEncLabelModel; print('Torch',torch.__version__,'GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); assert torch.cuda.is_available()"])
checked([PYTHON, "-m", "unittest", "test_overlap_activity", "test_sortformer_overlap"], cwd=WORK)


## Restore the completed DiaPer result bundle


In [ ]:
input_root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
result_root = None
for run_file in sorted(input_root.rglob("run-input.json")):
    try:
        run_input = json.loads(run_file.read_text())
    except Exception:
        continue
    candidate = run_file.parent
    required = [candidate/"full_video_evidence.json",
                candidate/"diaper-overlap"/"input"/"full-video.wav",
                candidate/"diaper-overlap"/"comparison.json"]
    if (run_input.get("video_id") == VIDEO_ID
            and run_input.get("notebook_revision") == "diaper-librosa-keywords-v23"
            and all(path.is_file() for path in required)):
        result_root = candidate
        break

# Local/manual ZIP fallback. Kaggle normally expands dataset archives, so the
# direct-file path above is the expected route.
if result_root is None:
    archives = sorted(input_root.rglob("diarization-results*.zip"))
    for archive in archives:
        with zipfile.ZipFile(archive) as zipped:
            run_names = [name for name in zipped.namelist() if name.endswith("run-input.json")]
            if len(run_names) != 1:
                continue
            run_input = json.loads(zipped.read(run_names[0]))
            if (run_input.get("video_id") != VIDEO_ID
                    or run_input.get("notebook_revision") != "diaper-librosa-keywords-v23"):
                continue
            if PRIOR.exists(): shutil.rmtree(PRIOR)
            PRIOR.mkdir()
            for member in zipped.infolist():
                target = (PRIOR/member.filename).resolve()
                if not target.is_relative_to(PRIOR.resolve()):
                    raise RuntimeError("Unsafe path in result ZIP")
            zipped.extractall(PRIOR)
            result_root = PRIOR
            break

if result_root is None:
    raise RuntimeError("Attach the Kaggle dataset created from completed diarization-results(15).zip.")
BASELINE = next(result_root.rglob("full_video_evidence.json"))
AUDIO = next(result_root.rglob("full-video.wav"))
DIAPER = next(result_root.rglob("diaper-overlap/comparison.json"))
print("Using extracted results:", result_root)
print("Baseline:", BASELINE)
print("Audio:", AUDIO)


## Run the head-to-head comparison

The 7:18 audio is divided into 80-second evaluation cores with five seconds of context on either side. Speaker labels remain local to each window; this test measures overlap activity rather than cross-window identity.


In [ ]:
log_path = OUTPUT/"sortformer.log"
command = [PYTHON, str(WORK/"run_sortformer_overlap.py"),
           "--audio", str(AUDIO), "--output-dir", str(OUTPUT/"inference")]
with log_path.open("w") as log:
    process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    recent = []
    for line in process.stdout:
        log.write(line); log.flush(); print(line, end="")
        recent = (recent + [line.rstrip()])[-30:]
    if process.wait() != 0:
        raise RuntimeError("Sortformer failed. Last output:\n" + "\n".join(recent))

comparison_path = OUTPUT/"comparison.json"
checked([PYTHON, str(WORK/"evaluate_overlap_activity.py"),
         "--baseline", str(BASELINE),
         "--rttm", str(OUTPUT/"inference"/"sortformer.rttm"),
         "--output", str(comparison_path)], cwd=WORK)
sortformer = json.loads(comparison_path.read_text())
diaper = json.loads(DIAPER.read_text())
head_to_head = {
    "revision": REVISION,
    "video_id": VIDEO_ID,
    "sortformer": sortformer["summary"],
    "diaper": diaper["summary"],
    "note": "Agreement with baseline overlap evidence is not ground-truth accuracy."
}
(OUTPUT/"head-to-head.json").write_text(json.dumps(head_to_head, indent=2)+"\n")
print(json.dumps(head_to_head, indent=2))


## Download the comparison


In [ ]:
with (OUTPUT/"runtime-packages.txt").open("w") as packages:
    checked([PYTHON, "-m", "pip", "freeze"], stdout=packages)
shutil.make_archive(str(BASE/"sortformer-comparison-results"), "zip", OUTPUT)
from IPython.display import FileLink, display
display(FileLink(str(BASE/"sortformer-comparison-results.zip")))
